In [3]:


# Step 1: Install and import libraries
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-image", "bokeh", "iqplot"])

import os
import numpy as np
import matplotlib.pyplot as plt
import skimage.io
import skimage.filters
import skimage.morphology
from urllib.request import urlretrieve
from IPython.display import display, Markdown

# Step 2: Download images
data_path = "https://s3.amazonaws.com/bebi103.caltech.edu/data/"
phase_file = "bsub_100x_phase.tif"
cfp_file = "bsub_100x_cfp.tif"

for filename in [phase_file, cfp_file]:
    if not os.path.exists(filename):
        urlretrieve(data_path + filename, filename)

# Step 3: Load phase contrast image
im_phase = skimage.io.imread(os.path.join(data_path if False else ".", phase_file))
ip_distance = 0.0626

print("Phase Contrast Image:")
print("The image is loaded as a NumPy array. Each value represents the intensity of an individual pixel.")
print("Image shape:", im_phase.shape)

# Step 4: Display phase contrast image
plt.figure(figsize=(7,5))
plt.imshow(im_phase, cmap='gray')
plt.title("Phase Contrast Image")
plt.xlabel("Pixels")
plt.ylabel("Pixels")
plt.colorbar(label="Intensity")
plt.show()

print("Explanation: The grayscale image shows Bacillus subtilis bacteria against a brighter background.")

# Step 5: Display image using different lookup tables
fig, axes = plt.subplots(2,2,figsize=(12,9))
colormaps = ['gray', 'magma', 'viridis', 'turbo']

for ax, cmap in zip(axes.flat, colormaps):
    ax.imshow(im_phase, cmap=cmap)
    ax.set_title(cmap.upper())
    ax.axis('off')

plt.suptitle("Different Lookup Tables (LUT)")
plt.tight_layout()
plt.show()

print("Explanation: Lookup tables change how pixel intensities are displayed using different colours. Viridis, magma, turbo and grayscale help visualize image details.")

# Step 6: Display histogram of phase image
plt.figure(figsize=(8,5))
plt.hist(im_phase.flatten(), bins=100)
plt.title("Histogram of Phase Contrast Image")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels")
plt.show()

print("Explanation: The histogram shows the distribution of pixel intensities. The smaller peak represents darker bacterial regions, while the larger peak represents the brighter background.")

# Step 7: Thresholding of phase image
thresh_phase = 300
im_phase_bw = im_phase < thresh_phase

fig, axes = plt.subplots(1,2,figsize=(12,5))
axes[0].imshow(im_phase, cmap='gray')
axes[0].set_title("Original Phase Image")
axes[1].imshow(im_phase_bw, cmap='gray')
axes[1].set_title("Thresholded Phase Image")

for ax in axes:
    ax.axis('off')

plt.show()

print("Explanation: Thresholding separates bacteria from the background. Pixels below intensity 300 are classified as bacteria, while the remaining pixels are classified as background.")

# Step 8: Overlay segmentation on phase image
im_phase_rgb = np.dstack(3 * [im_phase / im_phase.max()])
im_phase_rgb[im_phase_bw, 1] = 1.0

plt.figure(figsize=(8,6))
plt.imshow(im_phase_rgb)
plt.title("Phase Image Segmentation Overlay")
plt.axis('off')
plt.show()

print("Explanation: The green colour highlights the detected bacterial regions. Some unwanted background regions may also be detected.")

# Step 9: Load CFP image
im_cfp = skimage.io.imread(os.path.join(".", cfp_file))

plt.figure(figsize=(8,6))
plt.imshow(im_cfp, cmap='gray')
plt.title("Original CFP Image")
plt.axis('off')
plt.show()

print("Explanation: The CFP image uses fluorescence to make bacteria appear brighter than the background, making segmentation easier.")

# Step 10: Display a small region containing noisy pixels
plt.figure(figsize=(7,5))
plt.imshow(im_cfp[150:250, 450:550], cmap='gray')
plt.title("CFP Image - Noisy Pixel Region")
plt.colorbar(label="Intensity")
plt.show()

print("Explanation: The enlarged region shows unusually bright pixels caused by camera noise. These pixels can affect image analysis.")

# Step 11: Apply median filtering
selem = np.ones((3,3))
im_cfp_filt = skimage.filters.median(im_cfp, footprint=selem)

plt.figure(figsize=(8,6))
plt.imshow(im_cfp_filt, cmap='gray')
plt.title("Median Filtered CFP Image")
plt.axis('off')
plt.show()

print("Explanation: Median filtering removes unwanted noise by replacing each pixel with the median value of its neighbouring pixels.")

# Step 12: Display histogram of filtered CFP image
plt.figure(figsize=(8,5))
plt.hist(im_cfp_filt.flatten(), bins=100)
plt.title("Histogram of Filtered CFP Image")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels")
plt.show()

print("Explanation: The histogram shows pixel intensity distribution after noise removal. It helps identify a suitable threshold for bacterial segmentation.")

# Step 13: Display histogram using logarithmic scale
plt.figure(figsize=(8,5))
plt.hist(im_cfp_filt.flatten(), bins=100)
plt.yscale('log')
plt.ylim(1e2, 1e6)
plt.title("CFP Histogram - Logarithmic Scale")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels (Log Scale)")
plt.show()

print("Explanation: The logarithmic scale makes less frequent intensity values easier to observe. A threshold of approximately 140 can be selected.")

# Step 14: Thresholding in CFP channel
thresh_cfp = 140
im_cfp_bw = im_cfp_filt > thresh_cfp

fig, axes = plt.subplots(1,2,figsize=(12,5))
axes[0].imshow(im_cfp, cmap='gray')
axes[0].set_title("Original CFP Image")
axes[1].imshow(im_cfp_bw, cmap='gray')
axes[1].set_title("Thresholded CFP Image")

for ax in axes:
    ax.axis('off')

plt.show()

print("Explanation: Pixels with intensity greater than 140 are classified as bacteria. The white regions represent detected bacteria.")

# Step 15: Overlay CFP segmentation on phase image
im_rgb = np.dstack(3 * [im_phase / im_phase.max()])
im_rgb[im_cfp_bw, 1] = 1.0

plt.figure(figsize=(8,6))
plt.imshow(im_rgb)
plt.title("Bacterial Segmentation Overlay")
plt.axis('off')
plt.show()

print("Explanation: The green regions indicate bacteria detected using the CFP channel. Fluorescence provides better segmentation than phase contrast in this example.")

# Step 16: Apply Otsu's automatic thresholding
thresh_phase_otsu = skimage.filters.threshold_otsu(im_phase)
thresh_cfp_otsu = skimage.filters.threshold_otsu(im_cfp_filt)

print("\nOtsu Threshold Results")
print("Phase by eye:", thresh_phase)
print("CFP by eye:", thresh_cfp)
print("Phase by Otsu:", thresh_phase_otsu)
print("CFP by Otsu:", thresh_cfp_otsu)

print("\nExplanation: Otsu's method automatically calculates a threshold by analysing pixel intensity distribution. Its result can be compared with manually selected thresholds.")

# Step 17: Calculate bacterial area in pixels
bacterial_area_pix = (im_cfp_filt > thresh_cfp_otsu).sum()

print("\nBacterial Area in Pixels:")
print("Bacterial area =", bacterial_area_pix, "pixels")

print("Explanation: The total number of white pixels in the segmented CFP image represents the bacterial area in pixels.")

# Step 18: Calculate bacterial area in square microns
bacterial_area_micron = bacterial_area_pix * ip_distance**2

print("\nBacterial Area in Square Microns:")
print("Bacterial area =", bacterial_area_micron, "square microns")

print("Explanation: The pixel area is converted into square microns using the interpixel distance of 0.0626 microns.")


Output hidden; open in https://colab.research.google.com to view.